In [1]:
# Angka tetap. Setiap undian acak di notebook ini memakai SEED.
SEED = 42


# Kunci ujian tingkat intermediate

Jalankan dari atas ke bawah. Daftar soal di bawah sama dengan README dan `ujian.ipynb`. Kunci memakai nomor yang sama.

Pustaka adalah kumpulan fungsi siap pakai.

**Proyek mini.** Alarm jemuran pada data sintetis baru, `random_state=SEED`, tanpa unduhan. Kelas positif adalah hujan. Hujan lebih jarang. Melewatkan hujan membuat jemuran basah. Alarm palsu hanya membuat jemuran diangkat terlalu cepat. Baseline selalu menebak kelas yang paling banyak. Bandingkan baseline itu dengan pohon `max_depth=2` pada data uji, termasuk precision, recall, dan empat kotak. Bandingkan juga pohon itu dengan pohon tanpa batas kedalaman.

### Soal

1. Metrik mana yang diutamakan, dan mengapa?
2. Berapa recall baseline?
3. Berapa precision dan recall pohon pendek pada data uji?
4. Pohon mana yang menghafal? Bandingkan selisih skor data latih dan data uji antara pohon tanpa batas dan pohon pendek.
5. Model mana yang dilaporkan?
6. Jika rata-rata kolom dihitung dari seluruh baris sebelum data dibagi, apa nama kesalahannya, dan rata-rata mana yang boleh?

### Kunci ujian

1. Recall. Melewatkan hujan lebih merugikan daripada alarm palsu sesekali. Baris `ujian_metrik_dipilih recall`.
2. `ujian_recall_baseline 0.0`. Baseline tidak menemukan hujan.
3. Precision pohon pendek pada data uji `ujian_precision 0.8`. Recall pohon pendek pada data uji `ujian_recall 0.631578947368421`.
4. Pohon tanpa batas yang menghafal. Selisih skor data latih dan data uji pohon itu `ujian_selisih_dalam 0.10833333333333328`. Selisih pohon pendek `ujian_selisih_dangkal 0.04761904761904767`.
5. Pohon pendek yang dilaporkan. Baris `ujian_model_dipilih dangkal`.
6. Data leakage. Rata-rata yang boleh adalah rata-rata data latih. Baris `ujian_nama_kebocoran data leakage` dan `ujian_rata_boleh data latih`.

Soal 3, 4, dan 5 di atas sudah tetap: precision dan recall pada data uji, pohon tanpa batas yang menghafal, dan pohon pendek yang dilaporkan.

Empat kotak, skor latih, dan skor uji ada pada baris di bawah. Angka ini disalin dari keluaran yang dijalankan.

#### Angka yang disalin dari keluaran notebook

```
ujian_baris 400
ujian_bagian_hujan 0.16
ujian_n_latih 280
ujian_n_uji 120
ujian_cocok_baseline 0.8416666666666667
ujian_recall_baseline 0.0
ujian_skor_latih_dangkal 0.9642857142857143
ujian_skor_uji_dangkal 0.9166666666666666
ujian_tepat_negatif 98
ujian_salah_positif 3
ujian_salah_negatif 7
ujian_tepat_positif 12
ujian_precision 0.8
ujian_recall 0.631578947368421
ujian_skor_latih_dalam 1.0
ujian_skor_uji_dalam 0.8916666666666667
ujian_selisih_dalam 0.10833333333333328
ujian_selisih_dangkal 0.04761904761904767
ujian_menghafal dalam
ujian_model_dipilih dangkal
ujian_metrik_dipilih recall
ujian_nama_kebocoran data leakage
ujian_rata_boleh data latih
```


In [2]:
# Satu ide: panggil pustaka untuk kunci ujian.
# Pustaka adalah kumpulan fungsi siap pakai, bukan rak buku.
import numpy as np
from sklearn.datasets import make_classification
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


### Arti parameter

Menurut dokumentasi scikit-learn, make_classification mengundi soal klasifikasi.

n_samples adalah banyaknya baris.

n_features adalah banyaknya kolom semuanya: kolom informatif, kolom redundan, kolom salinan, dan sisanya kolom acak yang tidak berguna.

n_informative adalah banyaknya kolom informatif. Tiap kelas punya kelompok sebaran normal (simpangan baku 1) di sekitar sudut hypercube berdimensi sebanyak itu.

Hypercube adalah kotak bersudut di beberapa dimensi. Bukan istilah baru.

Di dalam tiap kelompok, kolom informatif diundi dari sebaran normal baku (rata-rata 0, simpangan baku 1), lalu digabung secara linear, sebelum kelompok diletakkan di sudut hypercube.

n_redundant adalah banyaknya kolom kombinasi linear acak dari kolom informatif.

n_repeated adalah banyaknya kolom salinan, diambil acak dari kolom informatif dan redundan.

n_clusters_per_class adalah banyaknya kelompok pada tiap kelas.

class_sep mengalikan ukuran hypercube. Panjang sisinya 2 kali class_sep. Makin besar, kelompok makin berjauhan dan soal makin mudah.

flip_y adalah bagian baris yang kelasnya diundi ulang secara acak.

weights adalah proporsi baris yang ditugaskan ke tiap kelas. Proporsi akhir tidak persis sama dengan weights bila flip_y bukan nol.

random_state membuat undian pembuatan data ini berulang selama angkanya sama.

Nama parameter ini bukan istilah baru.


In [3]:
# Satu ide: data ujian sintetis. Kelas hujan lebih sedikit.
X_ujian, y_ujian = make_classification(
    n_samples=400,
    n_features=20,
    n_informative=3,
    n_redundant=0,
    n_repeated=0,
    n_clusters_per_class=1,
    weights=[0.85, 0.15],
    class_sep=1.3,
    flip_y=0.02,
    random_state=SEED,
)
print("ujian_baris", int(X_ujian.shape[0]))
print("ujian_bagian_hujan", float(y_ujian.mean()))


ujian_baris 400
ujian_bagian_hujan 0.16


In [4]:
# Satu ide: sisihkan data uji. stratify menjaga proporsi kelas. Bukan istilah baru.
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X_ujian,
    y_ujian,
    test_size=0.3,
    random_state=SEED,
    shuffle=True,
    stratify=y_ujian,
)
print("ujian_n_latih", int(len(y_latih)))
print("ujian_n_uji", int(len(y_uji)))


ujian_n_latih 280
ujian_n_uji 120


In [5]:
# Satu ide: baseline selalu menebak kelas yang paling banyak.
kelas_banyak = int(np.bincount(y_latih).argmax())
tebakan_baseline = np.full_like(y_uji, kelas_banyak)
print("ujian_cocok_baseline", float((tebakan_baseline == y_uji).mean()))
assert np.array_equal(
    tebakan_baseline, np.full_like(y_uji, int(np.bincount(y_latih).argmax()))
)


ujian_cocok_baseline 0.8416666666666667


In [6]:
# Satu ide: recall baseline.
print(
    "ujian_recall_baseline",
    float(recall_score(y_uji, tebakan_baseline, pos_label=1, zero_division=0)),
)


ujian_recall_baseline 0.0


In [7]:
# Satu ide: latih pohon pendek dan cetak skor data latih.
pohon = DecisionTreeClassifier(max_depth=2, random_state=SEED)
pohon.fit(X_latih, y_latih)
assert int(pohon.tree_.n_node_samples[0]) == len(y_latih)
skor_latih_dangkal = float(pohon.score(X_latih, y_latih))
print("ujian_skor_latih_dangkal", skor_latih_dangkal)


ujian_skor_latih_dangkal 0.9642857142857143


In [8]:
# Satu ide: skor pohon pendek pada data uji.
tebakan_uji = pohon.predict(X_uji)
skor_uji_dangkal = float((tebakan_uji == y_uji).mean())
print("ujian_skor_uji_dangkal", skor_uji_dangkal)


ujian_skor_uji_dangkal 0.9166666666666666


In [9]:
# Satu ide: empat kotak pohon pendek pada data uji.
tn, fp, fn, tp = confusion_matrix(y_uji, tebakan_uji, labels=[0, 1]).ravel()
print("ujian_tepat_negatif", int(tn))
print("ujian_salah_positif", int(fp))
print("ujian_salah_negatif", int(fn))
print("ujian_tepat_positif", int(tp))


ujian_tepat_negatif 98
ujian_salah_positif 3
ujian_salah_negatif 7
ujian_tepat_positif 12


In [10]:
# Satu ide: precision pohon pendek pada data uji.
precision_dangkal = float(
    precision_score(y_uji, tebakan_uji, pos_label=1, zero_division=0)
)
print("ujian_precision", precision_dangkal)
assert precision_dangkal == tp / (tp + fp)


ujian_precision 0.8


In [11]:
# Satu ide: recall pohon pendek pada data uji.
recall_dangkal = float(
    recall_score(y_uji, tebakan_uji, pos_label=1, zero_division=0)
)
print("ujian_recall", recall_dangkal)
assert recall_dangkal == tp / (tp + fn)


ujian_recall 0.631578947368421


In [12]:
# Satu ide: skor pohon tanpa batas pada data latih.
pohon_dalam = DecisionTreeClassifier(max_depth=None, random_state=SEED)
pohon_dalam.fit(X_latih, y_latih)
assert int(pohon_dalam.tree_.n_node_samples[0]) == len(y_latih)
skor_latih_dalam = float(pohon_dalam.score(X_latih, y_latih))
print("ujian_skor_latih_dalam", skor_latih_dalam)


ujian_skor_latih_dalam 1.0


In [13]:
# Satu ide: skor dan recall pohon tanpa batas pada data uji.
skor_uji_dalam = float(pohon_dalam.score(X_uji, y_uji))
print("ujian_skor_uji_dalam", skor_uji_dalam)
tebakan_dalam = pohon_dalam.predict(X_uji)
_tn_dalam, _fp_dalam, fn_dalam, tp_dalam = confusion_matrix(
    y_uji, tebakan_dalam, labels=[0, 1]
).ravel()
ujian_recall_dalam = float(tp_dalam / (tp_dalam + fn_dalam))
print("ujian_recall_dalam", ujian_recall_dalam)
assert float(pohon_dalam.score(X_latih, y_latih)) == 1.0, (
    "Pohon tanpa batas harus menghafal data latih."
)
assert ujian_recall_dalam == float(
    recall_score(y_uji, tebakan_dalam, pos_label=1, zero_division=0)
), "Recall pohon dalam harus sama dengan hitungan kotak pada data uji."


ujian_skor_uji_dalam 0.8916666666666667
ujian_recall_dalam 0.631578947368421


In [14]:
# Satu ide: selisih skor menunjukkan pohon dalam yang menghafal.
selisih_dalam = float(skor_latih_dalam - skor_uji_dalam)
selisih_dangkal = float(skor_latih_dangkal - skor_uji_dangkal)
print("ujian_selisih_dalam", selisih_dalam)
print("ujian_selisih_dangkal", selisih_dangkal)
pohon_dalam_menghafal = selisih_dalam > selisih_dangkal
assert pohon_dalam_menghafal
menghafal = "dalam" if pohon_dalam_menghafal else "dangkal"
print("ujian_menghafal", menghafal)


ujian_selisih_dalam 0.10833333333333328
ujian_selisih_dangkal 0.04761904761904767
ujian_menghafal dalam


In [15]:
# Satu ide: pilih model dangkal dan metrik recall pada data uji.
recall_baseline = float(
    recall_score(y_uji, tebakan_baseline, pos_label=1, zero_division=0)
)
model_dipilih = (
    "dangkal"
    if menghafal == "dalam" and recall_dangkal > recall_baseline
    else "periksa_lagi"
)
print("ujian_model_dipilih", model_dipilih)
print("ujian_metrik_dipilih", "recall")
print("ujian_nama_kebocoran", "data leakage")
print("ujian_rata_boleh", "data latih")
assert menghafal == "dalam"
assert model_dipilih == "dangkal"
assert recall_dangkal > recall_baseline


ujian_model_dipilih dangkal
ujian_metrik_dipilih recall
ujian_nama_kebocoran data leakage
ujian_rata_boleh data latih
